In [ ]:
import pefile
from capstone import Cs, CS_ARCH_X86, CS_MODE_32, CS_MODE_64

input_path = r"C:\Path\To\program.exe"
output_path = r"C:\Path\To\text_section.asm"

pe = pefile.PE(input_path)

if pe.FILE_HEADER.Machine == 0x8664:
    mode = CS_MODE_64
elif pe.FILE_HEADER.Machine == 0x14C:
    mode = CS_MODE_32
else:
    raise ValueError("Unsupported architecture")

md = Cs(CS_ARCH_X86, mode)

with open(output_path, "w", encoding="utf-8") as out:
    for section in pe.sections:
        name = section.Name.decode(errors="ignore").strip("\x00")

        if name == ".text":
            code = section.get_data()
            address = pe.OPTIONAL_HEADER.ImageBase + section.VirtualAddress

            out.write(f"; Disassembly of {name}\n")
            out.write(f"; Start address: 0x{address:X}\n")
            out.write(f"; Size: {len(code)} bytes\n\n")

            for instruction in md.disasm(code, address):
                out.write(
                    f"0x{instruction.address:016X}:\t"
                    f"{instruction.mnemonic:<8}"
                    f"{instruction.op_str}\n"
                )

print(f"Done. Wrote full .text disassembly to {output_path}")